# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

**Google Colab:** use **Option B** (`%pip`) in the next section — skip the `uv` cell unless you know you need it. **Local (e.g. Cursor):** use [`uv`](https://github.com/astral-sh/uv) as below.

The `uv` steps:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

### Option B: Google Colab

Install dependencies into the **current** Colab runtime (no `uv` or extra kernel). Use a **GPU** runtime for vLLM. After this finishes, use **Runtime → Restart runtime** once, then continue.


In [ ]:
# Colab: skip the `uv` cell above when using this path.
#
# Plain `pip install vllm` often pulls a CUDA 13-built wheel (needs libcudart.so.13), which Colab does not provide on LD_LIBRARY_PATH.
# Use PyTorch cu129 + the official vLLM +cu129 release wheel so everything targets CUDA 12.9 (matches Colab drivers).
# GPU VMs are usually x86_64. On aarch64, replace `x86_64` with `aarch64` in the wheel URL.
#
%pip install -q sympy numpy tqdm "antlr4-python3-runtime==4.11.1"
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129 --upgrade
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0+cu129-cp38-abi3-manylinux_2_31_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip install -q transformers "bitsandbytes>=0.48.1"


In [ ]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install "bitsandbytes>=0.48.1" sympy numpy transformers vllm tqdm antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")


In [ ]:
# # activate venv after installation. This needs to be run everytime.
# !source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — results filename (default `results/starter_results.jsonl`). On **Colab**, after the Drive setup cell, saves go to **`My Drive/CSE151B/results/<filename>`**; locally, relative to the working directory
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [ ]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # Colab / single-GPU: must be "0". Use "1" only if a 2nd GPU exists.
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 4096                   # was 32768 — thinking models generate massive CoT, cap saves huge time

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
# V1 default starts a separate EngineCore process; that commonly fails in Jupyter/Colab.
# In-process mode avoids that (see vLLM LLMEngine.from_engine_args + envs.VLLM_ENABLE_V1_MULTIPROCESSING).
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import glob
import site

def _prepend_nvidia_libs_to_ld_path() -> None:
    roots = list(site.getsitepackages())
    user_site = getattr(site, "getusersitepackages", lambda: None)()
    if user_site:
        roots.append(user_site)
    libdirs: list[str] = []
    seen: set[str] = set()
    for root in roots:
        for d in glob.glob(os.path.join(root, "nvidia", "*", "lib")):
            if os.path.isdir(d) and d not in seen:
                seen.add(d)
                libdirs.append(d)
    if libdirs:
        sep = os.pathsep
        os.environ["LD_LIBRARY_PATH"] = sep.join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(sep)


_prepend_nvidia_libs_to_ld_path()

import io
import re
import sys
from pathlib import Path
from typing import Optional

# Jupyter: ipykernel replaces stdout with a stream that has no fileno(); vLLM workers need it.
try:
    sys.stdout.fileno()
except (io.UnsupportedOperation, AttributeError, OSError):
    sys.stdout = os.fdopen(1, "w", buffering=1)
    sys.stderr = os.fdopen(2, "w", buffering=1)

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

### Colab: put `public.jsonl` on Google Drive, then copy it here

The Colab VM does not include your git repo. Upload **`public.jsonl`** to  
`My Drive/CSE151B/data/public.jsonl` (or edit `DRIVE_JSONL` in the next cell), run the next cell, then load the dataset.


In [ ]:
# Google Colab — copy `public.jsonl` from Drive into `data/` (runtime has no repo files until you do).
# One-time: in Drive, create `CSE151B/data/` and upload `public.jsonl` there (from the assignment zip/repo).
# Change `DRIVE_JSONL` if you store the file elsewhere.

import shutil
from pathlib import Path

DRIVE_JSONL = Path("/content/drive/MyDrive/CSE151B/data/public.jsonl")
# Same assignment folder as data — put judger.py here for scoring.
DRIVE_PROJECT_ROOT = DRIVE_JSONL.parent.parent

try:
    from google.colab import drive
except ImportError:
    print("Skip: not Google Colab — use `data/public.jsonl` from your local repo clone.")
else:
    drive.mount("/content/drive")
    if not DRIVE_JSONL.is_file():
        raise FileNotFoundError(
            f"Missing on Drive: {DRIVE_JSONL}\n"
            "Upload public.jsonl to that path or set DRIVE_JSONL to your file."
        )
    Path("data").mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_JSONL, "data/public.jsonl")
    print(f"Copied to {Path('data/public.jsonl').resolve()}")


In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "Return ONLY the final answer inside \\boxed{}. "
    "Do not include reasoning, explanation, or extra text. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Return ONLY the letter inside \\boxed{}, e.g. \\boxed{C}. "
    "Do not include reasoning, explanation, or extra text."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=True,      # was False — reuses shared system prompt prefix across questions
    # enforce_eager removed           # was True — disabling CUDA graphs was a major bottleneck
    gpu_memory_utilization=0.88,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=64,
    max_num_batched_tokens=16384,    # was 8192 — raised to match max_model_len for better throughput
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

In [ ]:
#To install accelerate which is required to load model with transformers

#import sys
#print(sys.executable)
#!{sys.executable} -m pip show accelerate
#!{sys.executable} -m pip install -U accelerate

In [ ]:
# #Transformer version:

# from transformers import AutoTokenizer, AutoModelForCausalLM
# import torch

# MODEL_ID = "Qwen/Qwen3-4B"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     dtype=torch.float16,
#     device_map="auto",
#     trust_remote_code=True,
# )

# model.eval()
# print("Model loaded!")

In [ ]:
# import accelerate
# print(accelerate.__version__)

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [ ]:
import re
import json
from pathlib import Path

CHUNK_SIZE = 100

def extract_final_response(text):
    text = str(text).strip()

    # Extract \boxed{...}
    match = re.search(r"\\boxed\{([^{}]*)\}", text)
    if match:
        return r"\boxed{" + match.group(1).strip() + "}"

    # Fallback: use last non-empty line
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else text


try:
    _results_dir = DRIVE_PROJECT_ROOT / "results"
except NameError:
    _results_dir = Path(OUTPUT_PATH).parent

_results_dir.mkdir(parents=True, exist_ok=True)

# Use a NEW checkpoint name so old bad responses are not reused
response_checkpoint = _results_dir / (Path(OUTPUT_PATH).stem + ".clean.responses.jsonl")
print(f"Checkpoint path: {response_checkpoint}")

completed = {}

if response_checkpoint.exists():
    with open(response_checkpoint) as f:
        for line in f:
            r = json.loads(line)
            completed[r["id"]] = r["response"]
    print(f"Resumed from checkpoint: {len(completed)} responses already generated")

remaining = [d for d in data if d.get("id") not in completed]
print(f"Generating {len(remaining)} remaining / {len(data)} total...")

for chunk_start in range(0, len(remaining), CHUNK_SIZE):
    chunk = remaining[chunk_start : chunk_start + CHUNK_SIZE]

    prompts = []
    for item in chunk:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt_text)

    outputs = llm.generate(prompts, sampling_params=sampling_params)

    with open(response_checkpoint, "a") as f:
        for item, out in zip(chunk, outputs):
            raw_response = out.outputs[0].text.strip()
            response = extract_final_response(raw_response)

            completed[item["id"]] = response
            f.write(json.dumps({
                "id": item["id"],
                "response": response,
                "raw_response": raw_response
            }) + "\n")

    done = len(completed)
    print(f"  {done}/{len(data)}  ({done / len(data) * 100:.1f}%)")

responses = [completed[d["id"]] for d in data]
print(f"Done. {len(responses)} responses ready.")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
import os, sys, importlib

JUDGER_ROOT = "/content/drive/MyDrive/"  # folder containing judger.py and utils.py

sys.path.insert(0, JUDGER_ROOT)

for m in ["judger", "utils"]:
    if m in sys.modules:
        del sys.modules[m]

importlib.invalidate_caches()

from judger import Judger
judger = Judger(strict_extract=False)

print("Judger loaded from:", JUDGER_ROOT)

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


results = []

for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]

        correct = judger.auto_judge(
            pred=response,
            gold=gold_list,
            options=[[]] * len(gold_list),
        )

    results.append({
        "id": item.get("id"),
        "is_mcq": is_mcq,
        "gold": gold,
        "response": response,
        "correct": correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
import pandas as pd
from IPython.display import display

# Make sure these are created
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

summary = pd.DataFrame(
    [
        ["MCQ", sum(r["correct"] for r in mcq_res), len(mcq_res), f"{acc(mcq_res):.2f}%"],
        ["Free-form", sum(r["correct"] for r in free_res), len(free_res), f"{acc(free_res):.2f}%"],
        ["Overall", sum(r["correct"] for r in results), len(results), f"{acc(results):.2f}%"],
    ],
    columns=["Type", "Correct", "Total", "Accuracy"]
)

display(summary)

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
for _, row in summary.iterrows():
    print(f"{row['Type']:10s}: {row['Correct']:4d} / {row['Total']:4d}  ({row['Accuracy']})")
print("=" * 50)



In [ ]:
import pandas as pd
from IPython.display import display

# Recreate free_res from current results
free_res = [r for r in results if not r["is_mcq"]]

debug = pd.DataFrame([
    {
        "id": r["id"],
        "gold": repr(r["gold"]),
        "response": repr(r["response"])[:1000],
        "correct": r["correct"],
    }
    for r in free_res[:10]
])

display(debug)

## 9. Save Results

Results are written as newline-delimited JSON. The cell prints the **full path** when it runs.

**Where files go**

- **Google Colab** (after the Drive setup cell that defines `DRIVE_PROJECT_ROOT`): written to **`My Drive/CSE151B/results/`** (filename from `OUTPUT_PATH`, default `starter_results.jsonl`).
- **Local, or Colab without that cell:** `OUTPUT_PATH` relative to the working directory (default `results/starter_results.jsonl`).

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

# Colab: write straight to My Drive/CSE151B/results/ (requires Drive setup cell). Else use OUTPUT_PATH.

out_path = DRIVE_PROJECT_ROOT / "results" / Path(OUTPUT_PATH).name


out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path.resolve()}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!

In [ ]:
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    print("Not running in Colab — skipping runtime termination.")